In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import sqlite3

Loading the dataset 

In [15]:
df=pd.read_csv("train.csv")

Data cleaning 

In [13]:
df.shape
df.head()
df.info()
df.describe()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales'],
      dtype='str')

There are a datatpye error in df ,the columns order is and ship date has to in date datatypt ,so now i  will change the data type 

In [19]:
df[['Order Date','Ship Date']]=df[['Order Date','Ship Date']].apply(pd.to_datetime,format='mixed')
df.describe()

,Row ID,Order Date,Ship Date,Postal Code,Sales
count,9800.000000,9800,9800,9789.000000,9800.000000
mean,4900.500000,2017-04-12 14:24:35.265306,2017-04-21 19:45:12.489796,55273.322403,230.769059
min,1.000000,2015-01-02 00:00:00,2015-01-04 00:00:00,1040.000000,0.444000
25%,2450.750000,2016-05-02 12:00:00,2016-05-08 00:00:00,23223.000000,17.248000
50%,4900.500000,2017-05-30 00:00:00,2017-06-12 00:00:00,58103.000000,54.490000
75%,7350.250000,2018-04-11 00:00:00,2018-05-02 00:00:00,90008.000000,210.605000
max,9800.000000,2018-12-30 00:00:00,2019-05-01 00:00:00,99301.000000,22638.480000
std,2829.160653,NaN,NaN,32041.223413,626.651875


In [44]:
df.isnull().sum()
df.duplicated().sum()
# There is no duplicate value in df 


np.int64(0)

In [51]:
df.describe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9800 non-null   int64         
 1   Order ID       9800 non-null   str           
 2   Order Date     9800 non-null   datetime64[us]
 3   Ship Date      9800 non-null   datetime64[us]
 4   Ship Mode      9800 non-null   str           
 5   Customer ID    9800 non-null   str           
 6   Customer Name  9800 non-null   str           
 7   Segment        9800 non-null   str           
 8   Country        9800 non-null   str           
 9   City           9800 non-null   str           
 10  State          9800 non-null   str           
 11  Postal Code    9789 non-null   float64       
 12  Region         9800 non-null   str           
 13  Product ID     9800 non-null   str           
 14  Category       9800 non-null   str           
 15  Sub-Category   9800 non-null   s

In [27]:
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
order_details = pd.read_csv('order_details.csv')
products = pd.read_csv('products.csv')


conn = sqlite3.connect("superstore_sales.db")

customers.to_sql('customers',conn, if_exists='replace',index=False)
products.to_sql('products',conn,if_exists='replace',index=False)
orders.to_sql('orders',conn,if_exists='replace',index=False)
order_details.to_sql('order_details',conn,if_exists='replace',index=False)
print("database created successfully")


database created successfully


connecting sql 

In [29]:
#total orders 
q1="""
select count("order id") as total_orders  from orders
"""
pd.read_sql(q1,conn)

,total_orders
0,4922


 total_sales 

In [36]:
# Total sales 
q1=""" 
select sum(sales) as total_sales from order_details

"""
total_sales=pd.read_sql(q1,conn)

pd.set_option('display.float_format', '{:,.2f}'.format)
total_sales

,total_sales
0,"2,261,536.78"


In [ ]:
#year vise sales 
q1=""" 
select strftime('%Y',o."Order date") as year,
sum(od.sales)
from orders as o join order_details as od using("order id")
group by year 
order by year
"""
pd.read_sql(q1,conn)


,year,sum(od.sales)
0,2015,"479,856.21"
1,2016,"459,436.01"
2,2017,"600,192.55"
3,2018,"722,052.02"


In [10]:
q1 = """
SELECT COUNT(DISTINCT o."Order ID") as total_orders,
       COUNT(DISTINCT o."Customer ID") as total_customers,
       ROUND(SUM(od.sales),2) as total_sales
FROM order_details od
JOIN orders o ON od."Order ID" = o."Order ID"
"""
print(pd.read_sql(q1, conn))

   total_orders  total_customers  total_sales
0          4922              793   2261536.78


product wise sales 

In [58]:
q="""
select 
p.category,sum(od.sales) as total_sales
from products as p join order_details as od using("product Id")
group by p.category
"""
pd.read_sql(q,conn)

,Category,total_sales
0,Furniture,"728,658.58"
1,Office Supplies,"705,422.33"
2,Technology,"827,455.87"


Category Technology has highest sales 

.......................................sub category wise sales....................................................

In [ ]:
q="""
select 
p.category,
p.'sub-category',sum(od.sales) as total_sales
from products as p join order_details as od using("product Id")
group by p.'sub-category'
order by p.category
"""
sub_category_wise_sales=pd.read_sql(q,conn)

,Category,Sub-Category,total_sales
0,Furniture,Bookcases,"113,813.20"
1,Furniture,Chairs,"322,822.73"
2,Furniture,Furnishings,"89,212.02"
3,Furniture,Tables,"202,810.63"
4,Office Supplies,Appliances,"104,618.40"
5,Office Supplies,Art,"26,705.41"
6,Office Supplies,Binders,"200,028.79"
7,Office Supplies,Envelopes,"16,128.05"
8,Office Supplies,Fasteners,"3,001.96"
9,Office Supplies,Labels,"12,347.73"


Key Business Insights
Technology is a strong-performing category, with Phones generating the highest sales at 327,782.45.
Within the Furniture category, Chairs are the top-performing sub-category, generating 322,822.73 in sales.
In Office Supplies, Storage leads with 219,343.39 in sales, followed by Binders with 200,028.79.
The top-performing sub-categories overall are Phones (327,782.45) and Chairs (322,822.73), indicating strong customer demand for these products.
Fasteners is the lowest-performing sub-category, with only 3,001.96 in sales, followed by Labels (12,347.73) and Envelopes (16,128.05).
Phones and Chairs generate almost similar sales, with Phones slightly outperforming Chairs by approximately 4,960.
The business should focus on high-performing sub-categories such as Phones, Chairs, Storage, and Binders, while reviewing the demand, pricing, and inventory strategy for low-performing sub-categories.

...........................................................Top customers..................................................................

In [67]:
q="""
select
c."customer name" ,c.segment,c.state,c.city,sum(od.sales) as total_sales
from customers c  join orders o using("customer id") join order_details od using("order id")
group by  c."customer id",
    c."customer name",
    c.segment,
    c.state,
    c.city
order by total_sales desc limit 10
"""
top_customers=pd.read_sql(q,conn)
top_customers

,Customer Name,Segment,State,City,total_sales
0,Sean Miller,Home Office,North Carolina,Monroe,"25,043.05"
1,Tamara Chand,Corporate,Washington,Seattle,"19,052.22"
2,Raymond Buch,Consumer,New York,Auburn,"15,117.34"
3,Tom Ashbrook,Home Office,New York,New York City,"14,595.62"
4,Adrian Barton,Consumer,Arizona,Phoenix,"14,473.57"
5,Ken Lonsdale,Consumer,Illinois,Chicago,"14,175.23"
6,Sanjit Chand,Consumer,California,Concord,"14,142.33"
7,Hunter Lopez,Consumer,Texas,Houston,"12,873.30"
8,Sanjit Engle,Consumer,New York,New York City,"12,209.44"
9,Christopher Conant,Consumer,North Carolina,Fayetteville,"12,129.07"
